
# Chain-of-Thought Variants

We compare three CoT approaches:

1. **Few-shot CoT (Human-crafted examples):** Supply solved examples with reasoning.
2. **Zero-shot CoT:** Add *"Let's think step by step"* to the prompt.
3. **Auto-CoT:** Ask the LLM to generate reasoning examples for similar questions, then use them as few-shot CoT.

> Based on [LearnPrompting: Automatic Chain of Thought](https://learnprompting.org/docs/advanced/thought_generation/automatic_chain_of_thought)


In [4]:

from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate, FewShotPromptTemplate

from dotenv import load_dotenv
import os
load_dotenv()  # Ensures environment variables are loaded

from langchain_openai import ChatOpenAI 

llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.2)


## 1. Few-shot CoT (human-crafted examples)

In [5]:
example_prompt = PromptTemplate.from_template(
    """Question: {q}
Reasoning: {r}
Final answer: {a}
"""
)

examples = [
    {
        "q": "If you have 2 apples and buy 3 more, how many total?",
        "r": "Start with 2, add 3, gives 5.",
        "a": "5",
    },
    {
        "q": "Tom has 5 pens, gives away 2, then buys 1 more. How many now?",
        "r": "5-2=3, plus 1 = 4.",
        "a": "4",
    },
]

fs_template = FewShotPromptTemplate(
    examples=examples,
    example_prompt=example_prompt,
    suffix="Question: {user_q}\nReasoning:",
    input_variables=["user_q"],
)


def few_shot_cot(q: str):
    return (fs_template | llm).invoke({"user_q": q}).content

In [6]:
print(fs_template.invoke({"user_q": "test"}).text)

Question: If you have 2 apples and buy 3 more, how many total?
Reasoning: Start with 2, add 3, gives 5.
Final answer: 5


Question: Tom has 5 pens, gives away 2, then buys 1 more. How many now?
Reasoning: 5-2=3, plus 1 = 4.
Final answer: 4


Question: test
Reasoning:


In [7]:

q1 = "Alice had 4 oranges, bought 2 more, then ate 1. How many left?"
print(few_shot_cot(q1))


Start with 4, add 2 to get 6, then subtract 1 for the eaten orange, which gives 5.  
Final answer: 5


## 2. Zero-shot CoT

In [8]:

zs_cot_tpl = PromptTemplate.from_template(
    """Answer the following question. Let's think step by step.

Question: {q}
Reasoning:"""
)

def zero_shot_cot(q: str):
    return (zs_cot_tpl | llm).invoke({"q": q}).content

q2 = "A train moves at 50 km/h for 2 hours. How far does it travel?"
print(zero_shot_cot(q2))


To determine how far the train travels, we can use the formula for distance, which is:

\[ \text{Distance} = \text{Speed} \times \text{Time} \]

1. **Identify the speed of the train**: The train moves at a speed of 50 km/h.
2. **Identify the time the train travels**: The train travels for 2 hours.
3. **Plug the values into the formula**:

   \[
   \text{Distance} = 50 \, \text{km/h} \times 2 \, \text{hours}
   \]

4. **Calculate the distance**:

   \[
   \text{Distance} = 100 \, \text{km}
   \]

Therefore, the train travels **100 kilometers**.


## 3. Auto-CoT (LLM generates examples, then apply)

In [9]:

# Step 1: Generate reasoning examples automatically for similar tasks
gen_tpl = PromptTemplate.from_template(
    """Generate 3 examples of math word problems with reasoning and final answer.

Format:
[{{
    "q": "...",
    "r": "...",
    "a": "..."
}},
...]""")

generated = (gen_tpl | llm).invoke({}).content
print("Generated Examples:\n", generated)


Generated Examples:
 [
    {
        "q": "Sarah has 12 apples. She gives 3 apples to her friend and then buys 5 more apples. How many apples does she have now?",
        "r": "Sarah starts with 12 apples. After giving away 3 apples, she has 12 - 3 = 9 apples. Then, she buys 5 more apples, so she has 9 + 5 = 14 apples.",
        "a": "14"
    },
    {
        "q": "A car travels 60 miles per hour. How far will it travel in 2.5 hours?",
        "r": "To find the distance traveled, we use the formula: Distance = Speed × Time. Here, the speed is 60 miles per hour and the time is 2.5 hours. Therefore, the distance is 60 × 2.5 = 150 miles.",
        "a": "150"
    },
    {
        "q": "A rectangle has a length of 10 meters and a width of 4 meters. What is the area of the rectangle?",
        "r": "The area of a rectangle is calculated by multiplying the length by the width. Here, the length is 10 meters and the width is 4 meters. Thus, the area is 10 × 4 = 40 square meters.",
        "a": 

In [ ]:
# Step 2: Use generated examples as few-shot CoT for a new question

auto_examples = [
    {
        "q": "A farmer has 120 apples. He wants to pack them into boxes that can hold 15 apples each. How many boxes will he need?",
        "r": "To find out how many boxes are needed, divide the total number of apples by the number of apples each box can hold. 120 apples ÷ 15 apples/box = 8 boxes.",
        "a": "8",
    },
    {
        "q": "A school has 250 students. If 40% of the students are in the 6th grade, how many students are in the 6th grade?",
        "r": "To find the number of 6th grade students, calculate 40% of 250. This can be done by multiplying 250 by 0.40. 250 × 0.40 = 100 students.",
        "a": "100",
    },
    {
        "q": "A car travels 60 miles per hour. How far will it travel in 3 hours?",
        "r": "To find the distance traveled, multiply the speed of the car by the time traveled. 60 miles/hour × 3 hours = 180 miles.",
        "a": "180",
    },
]

auto_template = FewShotPromptTemplate(
    examples=auto_examples,
    example_prompt=example_prompt,
    suffix="Question: {user_q}\nReasoning:",
    input_variables=["user_q"],
)


def auto_cot(q: str):
    return (auto_template | llm).invoke({"user_q": q}).content


q3 = "John had 12 cookies, gave 5 to friends, then bought 3 more. How many now?"
print(auto_cot(q3))

To find out how many cookies John has now, start with the number of cookies he initially had, subtract the number he gave away, and then add the number he bought. 

1. Start with 12 cookies.
2. Subtract the 5 cookies he gave to friends: 12 - 5 = 7 cookies.
3. Add the 3 cookies he bought: 7 + 3 = 10 cookies.

Final answer: 10


In [13]:
# Step 1: Generate reasoning examples automatically for similar tasks
gen_tpl = PromptTemplate.from_template(
    """I have bought 3 kg of Rice, 4 kg of dhal, 3 packets of biscuits, 2 kg of sugar.

Format this as a list of json objects
"""
)

generated = (gen_tpl | llm).invoke({}).content
print("Generated Examples:\n", generated)

Generated Examples:
 Here is the information formatted as a list of JSON objects:

```json
[
    {
        "item": "Rice",
        "quantity": "3 kg"
    },
    {
        "item": "Dhal",
        "quantity": "4 kg"
    },
    {
        "item": "Biscuits",
        "quantity": "3 packets"
    },
    {
        "item": "Sugar",
        "quantity": "2 kg"
    }
]
```
